In [1]:
from util import import_ragas_custom, load_env_variables_from_all_env_files

import_ragas_custom('ragas_custom_2')
load_env_variables_from_all_env_files()

Arquivos copiados com sucesso!


In [2]:
import os
import asyncio
import nest_asyncio

from ragas.integrations.llama_index import evaluate
from ragas.run_config import RunConfig
from ragas.testset.synthesizers.testset_schema import Testset
from llama_index.llms.openai import OpenAI
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.openai import OpenAIEmbedding, OpenAIEmbeddingModelType
from llama_index.embeddings.ollama import OllamaEmbedding
from ragas.prompt.mixin import PromptMixin
from ragas.run_config import RunConfig

from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    Settings,
)

from ragas.metrics import (
    context_precision,
    context_recall,
    context_entity_recall,
    NoiseSensitivity,
    ResponseRelevancy,
    answer_relevancy,
    faithfulness,
    FactualCorrectness,
    SemanticSimilarity,
    NonLLMStringSimilarity,
    RougeScore,
    StringPresence,
    ExactMatch
)

from ragas.metrics._aspect_critic import SUPPORTED_ASPECTS

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\jmess\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [3]:
nest_asyncio.apply()

In [4]:
DATA_PATH = 'data'
TESTSET = 'testset_openai_4omini.jsonl'
PERSIST_DIR = "./storage"
LANGUAGE = 'portuguese'
TIMEOUT = 2400
RESULT_CSV = 'result_gpt4omini_gpt4omini_1.csv'
MODEL = 'llama3.2:3b'
MODEL_GPT = 'gpt-4o-mini-2024-07-18'
CACHE_DIR = 'cache_2'

In [5]:
run_config = RunConfig(timeout=TIMEOUT)

In [6]:
testset = Testset.from_jsonl(TESTSET).to_pandas()

print("Tamanho do dataset: ", len(testset))
print(testset.head())

Tamanho do dataset:  132
                                          user_input  \
0  Quais são as aplicações e benefícios dos No-Br...   
1  Quais são as aplicações e benefícios dos No-Br...   
2  impacto da proteção contra distúrbios de energ...   
3  De que forma a protecão contra distúrbios de e...   
4  implicações suporte técnico eficaz melhoria ex...   

                                  reference_contexts  \
0  [A EMPRESA\nOs    No-Breaks   da   CM     Coma...   
1  [A EMPRESA\nOs    No-Breaks   da   CM     Coma...   
2  [plicações de  \nmissão crítica, nas mais vari...   
3  [plicações de  \nmissão crítica, nas mais vari...   
4  [supor te técnico pode oferecer ., supor te té...   

                                           reference          synthesizer_name  
0  Os No-Breaks da CM Comandos são indicados para...  AbstractQuerySynthesizer  
1  Os No-Breaks da CM Comandos são indicados para...  AbstractQuerySynthesizer  
2  A proteção contra distúrbios de energia elétri...  Abst

In [7]:
nan_rows = testset[testset.isna().any(axis=1)]

print("Quantidade de nulos: ", len(nan_rows))
print(nan_rows)

del nan_rows

Quantidade de nulos:  0
Empty DataFrame
Columns: [user_input, reference_contexts, reference, synthesizer_name]
Index: []


In [8]:
testset = testset.dropna()

In [9]:
print("Quantidade de linhas após remoção de nulos: ", len(testset))
print(testset)

Quantidade de linhas após remoção de nulos:  132
                                            user_input  \
0    Quais são as aplicações e benefícios dos No-Br...   
1    Quais são as aplicações e benefícios dos No-Br...   
2    impacto da proteção contra distúrbios de energ...   
3    De que forma a protecão contra distúrbios de e...   
4    implicações suporte técnico eficaz melhoria ex...   
..                                                 ...   
127  Qual é a importância do termo 'BEN' no context...   
128  Qual é a função do disjuntor no processo de ma...   
129  Quais são as aplicações mais modernas do DSP n...   
130  Quais são as vantagens do modelo cliente-serve...   
131                        significado de 'Po' lti-S.O   

                                    reference_contexts  \
0    [A EMPRESA\nOs    No-Breaks   da   CM     Coma...   
1    [A EMPRESA\nOs    No-Breaks   da   CM     Coma...   
2    [plicações de  \nmissão crítica, nas mais vari...   
3    [plicações de  \n

In [10]:
min = 0
max = len(testset) // 3
testset = Testset.from_pandas(testset[min:max])

In [11]:
embeding = OpenAIEmbedding(model=OpenAIEmbeddingModelType.TEXT_EMBED_3_SMALL)
model = OpenAI(model=MODEL_GPT)
# embeding = OllamaEmbedding(model_name=MODEL)
# model = Ollama(model=MODEL)

Settings.embed_model = embeding
Settings.llm = model

In [12]:
metrics = [
    context_precision,
    context_recall,
    context_entity_recall,
    NoiseSensitivity(),
    ResponseRelevancy(),
    answer_relevancy,
    faithfulness,
    FactualCorrectness(),
    SemanticSimilarity(),
    NonLLMStringSimilarity(),
    RougeScore(),
    StringPresence(),
    ExactMatch()
]

metrics.extend(SUPPORTED_ASPECTS)

for query in metrics:
    if isinstance(query, PromptMixin):
        path = os.path.join(CACHE_DIR, query.__class__.__name__)
        if not os.path.exists(path):
            os.makedirs(path)

        try:
            prompts = query.load_prompts(path, LANGUAGE,)
            query.set_prompts(**prompts)
        except Exception:
            prompts = asyncio.run(query.adapt_prompts(LANGUAGE, None, True, True))
            query.set_prompts(**prompts)
            query.save_prompts(path)
            prompts = query.load_prompts(path, LANGUAGE)
            query.set_prompts(**prompts)


In [13]:
documents = SimpleDirectoryReader(DATA_PATH).load_data()
index = VectorStoreIndex.from_documents(documents, show_progress=True)
query_engine = index.as_query_engine(request_timeout=TIMEOUT)

Parsing nodes:   0%|          | 0/58 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/82 [00:00<?, ?it/s]

In [14]:
result = evaluate(
    query_engine=query_engine,
    metrics=metrics,
    dataset=testset,
    llm=model,
    embeddings=embeding,
    run_config=run_config
)

Running Query Engine:   0%|          | 0/44 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/792 [00:00<?, ?it/s]

In [15]:
result_dataframe = result.to_pandas()
result_dataframe.to_csv(RESULT_CSV)

In [16]:
print(result)

{'context_precision': 0.6364, 'context_recall': 0.5530, 'context_entity_recall': 0.0484, 'noise_sensitivity_relevant': 0.3566, 'answer_relevancy': 0.7713, 'faithfulness': 0.5649, 'factual_correctness': 0.2836, 'semantic_similarity': 0.8326, 'non_llm_string_similarity': 0.2324, 'rouge_score': 0.2694, 'string_present': 0.0000, 'exact_match': 0.0000, 'harmfulness': 0.5455, 'maliciousness': 0.7273, 'coherence': 1.0000, 'correctness': 0.9091, 'conciseness': 0.9545}
